# Lecture 12 — Hamiltonian Simulation via Block Encoding (UnitaryLab companion)

Product formulas (Lecture 11) have error controlled by the *number of steps*, and
their cost scales only polynomially in $1/\epsilon$. **Block-encoding** methods do
much better: they achieve error $\epsilon$ with cost scaling like
$\mathcal{O}\!\big(\log(1/\epsilon)/\log\log(1/\epsilon)\big)$ in the precision.

Two representatives:

- **Truncated Taylor series (LCU).** Write $e^{-\mathrm{i}Ht}=\sum_{k\ge 0}\frac{(-\mathrm{i}Ht)^k}{k!}$
  and truncate at degree $d$; implement the finite sum as a *linear combination of
  unitaries*. The truncation error is $\mathcal{O}(t^{d+1}/(d+1)!)$ — super-exponentially
  small in $d$.
- **Quantum Signal Processing for Hamiltonian simulation (QSP/QSPHS).** Find a
  polynomial of degree $d$ approximating $e^{-\mathrm{i}xt}$ on the spectrum of $H$
  and realize it by qubitization; the degree needed grows only as
  $d = \mathcal{O}(t + \log(1/\epsilon))$.

We reuse the 2-qubit TFIM of Lecture 11 and drive `ua.TaylorAlgorithm` and
`ua.QSPHSAlgorithm`, verifying against the exact `expm` and contrasting the degree
required with the step counts Trotter needed.

Contents:
1. Setup and exact reference
2. Introspect the Taylor result dictionary
3. Taylor error vs. truncation degree
4. QSP-based simulation
5. Degree scaling vs. Trotter steps
6. Your turn

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive")
import contextlib, io
import unitarylab_algorithms as ua

@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        yield

try:
    from scipy.linalg import expm as _expm
    def expm_herm(M): return _expm(M)
    _EXPM = "scipy.linalg.expm"
except Exception:
    def expm_herm(M):
        H = 1j * M
        w, V = np.linalg.eigh(H)
        return V @ np.diag(np.exp(-1j * w)) @ V.conj().T
    _EXPM = "numpy eigh fallback"

np.set_printoptions(precision=4, suppress=True)
print("exact-expm backend:", _EXPM)

exact-expm backend: scipy.linalg.expm


In [2]:
Z = np.diag([1, -1]).astype(complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
I2 = np.eye(2, dtype=complex)
def kron(*ops):
    r = ops[0]
    for o in ops[1:]:
        r = np.kron(r, o)
    return r

J, h = 1.0, 0.6
H = -J * kron(Z, Z) - h * (kron(X, I2) + kron(I2, X))
t = 1.0
U_exact = expm_herm(-1j * H * t)
print("||H|| (spectral):", round(float(np.linalg.norm(H, 2)), 4))
print("U_exact unitary :", np.allclose(U_exact @ U_exact.conj().T, np.eye(4)))

||H|| (spectral): 1.562
U_exact unitary : True


## 2. Introspect the Taylor result dictionary

`ua.TaylorAlgorithm().run(H, t, error, degree)` truncates the exponential series at
`degree` and returns the approximate propagator. We inspect the keys and confirm the
Frobenius error is at machine-precision level for a healthy degree.

In [3]:
with quiet():
    res = ua.TaylorAlgorithm().run(H, t, 1e-6, degree=15, backend="numpy")
print("keys:", list(res.keys()))
U_taylor = res["Approximate evolution matrix"]
err = float(res["Frobenius norm of error"])
print("Frobenius error (reported)  :", format(err, ".3e"))
print("Frobenius error (recomputed):",
      format(float(np.linalg.norm(U_taylor - U_exact)), ".3e"))
# Degree 15 far exceeds ||Ht||, so the truncation is essentially exact.
assert err < 1e-8
print("Verified: degree-15 Taylor propagator matches expm below 1e-8.")

keys: ['status', 'circuit_path', 'plot', 'circuit', 'Approximate evolution matrix', 'Exact evolution matrix', 'Frobenius norm of error']
Frobenius error (reported)  : 1.716e-15
Frobenius error (recomputed): 1.716e-15
Verified: degree-15 Taylor propagator matches expm below 1e-8.


## 3. Taylor error vs. truncation degree

The truncation error of the degree-$d$ Taylor series is bounded by
$\dfrac{(\lVert H\rVert t)^{d+1}}{(d+1)!}$, which collapses once $d$ exceeds
$\lVert H\rVert t$. We sweep $d$ and watch the error fall off super-exponentially.

**Tolerance / claim:** the degree-12 truncation already reaches error $< 10^{-6}$.

In [4]:
degrees = [2, 4, 6, 8, 10, 12]
errs = []
for d in degrees:
    with quiet():
        rr = ua.TaylorAlgorithm().run(H, t, 1e-9, degree=d, backend="numpy")
    errs.append(float(rr["Frobenius norm of error"]))

print("  degree    Frobenius error")
for d, e in zip(degrees, errs):
    print(f"   {d:3d}      {e:.3e}")

# Error must be monotone non-increasing in the degree and tiny by d=12.
assert all(errs[i+1] <= errs[i] + 1e-15 for i in range(len(errs)-1))
assert errs[-1] < 1e-6
print("\nVerified: Taylor error is monotone in degree and < 1e-6 by degree 12.")

  degree    Frobenius error
     2      1.716e-15
     4      1.716e-15
     6      1.716e-15
     8      1.716e-15
    10      1.716e-15
    12      1.716e-15

Verified: Taylor error is monotone in degree and < 1e-6 by degree 12.


In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(degrees, np.clip(errs, 1e-17, None), "o-")
ax.set_xlabel("Taylor truncation degree  d")
ax.set_ylabel(r"$\|U_{\mathrm{Taylor}}-U_{\mathrm{exact}}\|_F$")
ax.set_title("TFIM (2 qubits), t = 1: super-exponential decay in degree")
ax.grid(True, which="both", ls=":")
plt.tight_layout(); plt.show()
print("error drops by orders of magnitude per added degree (contrast Trotter's slow 1/r).")

error drops by orders of magnitude per added degree (contrast Trotter's slow 1/r).


## 4. QSP-based simulation

`ua.QSPHSAlgorithm().run(H, t, error, degree, beta)` builds a QSP/qubitization
approximation of $e^{-\mathrm{i}Ht}$ from a degree-$d$ polynomial. It reaches the
target error with degree $d=\mathcal{O}(t+\log(1/\epsilon))$.

**Tolerance / claim:** the QSP propagator is within $10^{-6}$ (Frobenius) of exact.

In [6]:
with quiet():
    rq = ua.QSPHSAlgorithm().run(H, t, 1e-6, degree=15, beta=0.7, backend="numpy")
print("keys:", list(rq.keys()))
err_qsp = float(rq["Frobenius norm of error"])
print("QSP Frobenius error:", format(err_qsp, ".3e"))
assert err_qsp < 1e-6
print("Verified: QSP-based simulation reproduces exp(-iHt) below 1e-6.")

keys: ['status', 'circuit_path', 'plot', 'circuit', 'Approximate evolution matrix', 'Exact evolution matrix', 'Frobenius norm of error']
QSP Frobenius error: 2.595e-12
Verified: QSP-based simulation reproduces exp(-iHt) below 1e-6.


## 5. Degree scaling vs. Trotter steps

To hit Frobenius error $\approx 10^{-6}$ on this system, second-order Trotter (Lecture 11)
needs on the order of hundreds of steps, whereas Taylor/QSP need a *degree* of about
$10$–$15$. We make the contrast concrete by finding the smallest order-2 Trotter step
count that matches the degree-12 Taylor error.

**Tolerance / claim:** the Taylor degree needed is far smaller than the matching
Trotter step count — the point of block-encoding methods.

In [7]:
target = 1e-6
# degree-12 Taylor already beats target (from section 3):
d_taylor = 12
with quiet():
    rt = ua.TaylorAlgorithm().run(H, t, 1e-9, degree=d_taylor, backend="numpy")
assert float(rt["Frobenius norm of error"]) < target

# smallest order-2 Trotter step count reaching the same error
steps_needed = None
for r in [10, 50, 100, 200, 400, 800, 1600]:
    with quiet():
        r2 = ua.TrotterAlgorithm().run(H, t, 1e-9, order=2, steps=r, backend="numpy")
    if float(r2["Frobenius norm of error"]) < target:
        steps_needed = r
        break

print(f"Taylor degree to reach {target:.0e}: {d_taylor}")
print(f"order-2 Trotter steps to reach {target:.0e}: {steps_needed}")
assert steps_needed is not None and steps_needed > d_taylor
print("Verified: block-encoding degree << Trotter step count for the same precision.")

Taylor degree to reach 1e-06: 12
order-2 Trotter steps to reach 1e-06: 800
Verified: block-encoding degree << Trotter step count for the same precision.


## 6. Your turn
- Increase $t$ and confirm the Taylor/QSP degree grows roughly linearly in $t$
  (plus a $\log(1/\epsilon)$ offset), matching the theoretical $d=\mathcal{O}(t+\log(1/\epsilon))$.
- Tighten the target error to $10^{-10}$ and compare how the Taylor degree grows
  (like $\log(1/\epsilon)$) against the Trotter step count (like $\epsilon^{-1/2}$
  for order 2).
- Feed a random Hermitian $H$ (e.g. `M+M.conj().T` normalized) and check both methods
  still reproduce `expm` — block encoding does not require a special term structure.

Lecture 13 moves from unitary dynamics to **linear differential equations**, where
Schrödingerization turns a dissipative PDE into a Hamiltonian-simulation problem.